# 03 · Filter & Rank — run the shared multi-layer filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 01** you're both a *user* (run it on your predictions) and its *author*
(P4 hardens its cutoffs from your calibration).

Run `00`–`02` first so `results/predictions.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)

## Build `Design` objects from your predictions

The filter operates on `fp.Design` records. Map your per-tool predictions onto its fields. For a
pure-confidence benchmark you may not have `designed_pdb`/`predicted_pdb` paths for every item — set
the metrics you do have; `self_consistency()` uses provided `scrmsd` if present.

In [ ]:
pred = pd.read_csv("results/predictions.csv")

# Use AF2 if available, else fall back to whatever tool you ran (mock for the dry run).
def pick_tool(g):
    for t in ("af2", "esmfold", "boltz", "mock"):
        if t in set(g["tool"]):
            return g[g["tool"] == t].iloc[0]
    return g.iloc[0]

designs = []
for _id, g in pred.groupby("id"):
    row = pick_tool(g)
    designs.append(fp.Design(
        design_id=str(_id),
        sequence="",                       # not needed for the confidence layers
        design_type="monomer",
        plddt=row.get("plddt"),
        pae_interaction=row.get("pae"),
        # scrmsd=...,                       # fill once you compute design-vs-predicted Cα-RMSD
        extra={"outcome": row.get("outcome")},
    ))
print(len(designs), "Design objects built")

## Run the pipeline

`run_pipeline()` applies the layers in order and returns a ranked DataFrame. With only pLDDT/PAE
present (no scRMSD yet), expect the self-consistency layer to be lenient — that's fine for the dry
run; the point of Project 01 is to *calibrate* these cutoffs in notebook 04.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="monomer")
df_ranked.to_csv("results/ranked.csv", index=False)
fp.report(df_ranked, top_n=10, save_prefix="results/proj01")
df_ranked.head(10)

## Survival-at-each-layer (honest accounting)

Report how many designs pass each layer. For Project 01 the more important table comes next
(metric vs *known outcome*), but the survival view is the standard cohort artifact.

In [ ]:
import pandas as pd
if "layers_passed" in df_ranked:
    print(df_ranked["layers_passed"].value_counts().sort_index())

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (not a one-off script).
- [ ] Survival-at-each-layer reported.
- [ ] Any mapping assumptions (which tool, which fields) written down.

**Next:** `04_validate.ipynb` — the calibration study (ROC/PR, composite, agreement).